In [ ]:
import dijido
from datetime import timedelta
from datetime import datetime
import pandas
import numpy
from plotly import graph_objects
from plotly import offline as plotly

import scipy
from scipy import signal

In [ ]:
# Start date is inclusive, and begins at midnight the day of the start date
START_DATE = "2023-02-01"

# End date is inclusive, and includes all time up to midnight of the following day
END_DATE = "2023-02-28"

In [ ]:
dijido.login("dibidave")

In [ ]:
start_date = datetime.strptime(START_DATE, "%Y-%m-%d")
date = start_date
end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

goals_by_date = {}

while date <= end_date:
    
    date_string = date.strftime("%Y-%m-%d")
    print(date_string)
    
    date_goals = dijido.get_goal_times_by_date_range(date_string, date_string, split_overlapping=True)
    goals_by_date[date_string] = date_goals
    
    date = date + timedelta(days=1)

In [ ]:
def get_children_goal_ids(goal_id):
    
    children_goal_ids = set([])
    
    for date in goals_by_date:
        for goal_name, goal in goals_by_date[date].items():
            if "parent_goal_ids" not in goal:
                continue
            if goal_id in goal["parent_goal_ids"]:
                children_goal_ids.add(goal["_id"])
                
    return children_goal_ids

In [ ]:
def plot_time_trace(goal_name):

    traces = []

    date = datetime.strptime(START_DATE, "%Y-%m-%d")
    end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

    xs = []
    ys = []

    while date <= end_date:

        days = (date - start_date).days
        xs.append(days)

        y = None

        date_string = date.strftime("%Y-%m-%d")

        if goal_name in goals_by_date[date_string]:
            y = goals_by_date[date_string][goal_name]["duration"] / 60 / 60
        elif not y:
            y = 0

        ys.append(y)

        date = date + timedelta(days=1)

    trace = graph_objects.Scatter(
        x=xs,
        y=ys,
        name=goal_name
    )

    traces.append(trace)

    layout = {
        "xaxis": {
            "tickmode": "array",
            "tickvals": xs,
            "ticktext": [(start_date + timedelta(days=x)).strftime("%a %m/%d") for x in xs]
        },
        "yaxis": {
            "title": "Hours per day"
        },
        "plot_bgcolor": "rgba(255, 255, 255, 0)",
        "paper_bgcolor": "rgba(255, 255, 255, 0)",
        "title": goal_name
    }

    figure = graph_objects.Figure(data=traces, layout=layout)

    plotly.iplot(figure)

In [ ]:
plot_time_trace("Procure, prepare, and consume food")

In [ ]:
def plot_time_traces_hierarchical(parent_goal_id, level=1):
    
    if level > num_levels:
        return
    
    child_goal_ids = get_children_goal_ids(parent_goal_id)
    
    if len(child_goal_ids) <= 1:
        return
    
    parent_goal_name = dijido.get_goal_by_id(parent_goal_id)["name"]

    traces = []
    data = []

    for child_goal_id in child_goal_ids:

        date = datetime.strptime(START_DATE, "%Y-%m-%d")
        end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

        xs = []
        ys = []

        child_goal = dijido.get_goal_by_id(child_goal_id)

        while date <= end_date:

            days = (date - start_date).days
            xs.append(days)

            y = None

            date_string = date.strftime("%Y-%m-%d")

            if child_goal_id in goals_by_date[date_string]:
                y = goals_by_date[date_string][child_goal_id]["duration"] / 60 / 60
            elif not y:
                y = 0

            ys.append(y)

            date = date + timedelta(days=1)

        trace = graph_objects.Scatter(
            x=xs,
            y=ys,
            name=child_goal["name"],
    #         line={'shape': 'spline', 'smoothing': 1.3}
        )
        
        data.append({
            "name": child_goal["name"],
            "mean (h/d)": numpy.mean(ys),
            "mean (h/w)": numpy.mean(ys)*7,
        })

        traces.append(trace)

    layout = {
        "xaxis": {
            "tickmode": "array",
            "tickvals": xs,
            "ticktext": [(start_date + timedelta(days=x)).strftime("%a %m/%d") for x in xs]
        },
        "yaxis": {
            "title": "Hours per day"
        },
        "plot_bgcolor": "rgba(255, 255, 255, 0)",
        "paper_bgcolor": "rgba(255, 255, 255, 0)",
        "title": parent_goal_name
    }

    figure = graph_objects.Figure(data=traces, layout=layout)

    plotly.iplot(figure)
    
    df = pandas.DataFrame.from_records(data)
    display(df.sort_values(by="mean (h/d)", ascending=False))
    
    for child_goal_id in child_goal_ids:
        plot_time_traces_hierarchical(child_goal_id, level+1)

In [ ]:
num_levels = 3
top_level_goal_name = "Optimize objective function (lol how meta)"
top_level_goal_id = dijido.get_goals_by_name(top_level_goal_name)[0]["_id"]
        
        
plot_time_traces_hierarchical(top_level_goal_id)

In [ ]:
from scipy.stats import gaussian_kde
from scipy import interpolate

In [ ]:
goals_of_interest = ["Untracked", "Change the world", "Experience life", "Procure, prepare, and consume food", "Establish and maintain connections with people", "Sleep", "Reduce risk of disease"]
traces = []
data = []

for goal_name in goals_of_interest:
    
    date = datetime.strptime(START_DATE, "%Y-%m-%d")
    end_date = datetime.strptime(END_DATE, "%Y-%m-%d")

    xs = []
    ys = []

    while date <= end_date:

        days = (date - start_date).days
        xs.append(days)

        y = None

        date_string = date.strftime("%Y-%m-%d")

        if goal_name in goals_by_date[date_string]:
            y = goals_by_date[date_string][goal_name]["duration"] / 60 / 60
        else:
            
            try:
                goals = dijido.get_goals_by_name(goal_name)
                goal_id = goals[0]["_id"]
                
                if goal_id in goals_by_date[date_string]:
                    y = goals_by_date[date_string][goal_id]["duration"] / 60 / 60
                else:
                    y = 0
            except:
                y = 0

        ys.append(y)

        date = date + timedelta(days=1)

    trace = graph_objects.Scatter(
        x=xs,
        y=ys,
        name=goal_name,
        line={
            "width": 3
        }
    )

    data.append({
        "name": goal_name,
        "mean (h/d)": numpy.mean(ys),
        "mean (h/w)": numpy.mean(ys)*7,
    })

    traces.append(trace)

layout = {
    "xaxis": {
        "tickmode": "array",
        "tickvals": xs,
        "ticktext": [(start_date + timedelta(days=x)).strftime("%a %m/%d") for x in xs]
    },
    "yaxis": {
        "title": "Hours per day"
    },
    "plot_bgcolor": "rgba(255, 255, 255, 0)",
    "paper_bgcolor": "rgba(255, 255, 255, 0)",
    "title": "Time analysis",
    "width": 1000
}



figure = graph_objects.Figure(data=traces, layout=layout)

plotly.iplot(figure)

df = pandas.DataFrame.from_records(data)
display(df.sort_values(by="mean (h/d)", ascending=False))